In [ ]:
# ========== 1. IMPORTS ========== #
import os, csv, time, random, shutil, glob, gc
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, DepthwiseConv2D, Activation, Input, Layer, Add,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"GPUs: {len(gpus)} | Policy: mixed_float16")

In [ ]:
# ========== 2. CONFIGURATION ========== #

STRATEGY_KEY   = "efsda_v3_crossbranch_multiscale"
STRATEGY_LABEL = "EfficientNetB4 + E-FSDA v3 (CrossBranch + MultiScale)"

DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

INPUT_SHAPE     = (380, 380, 3)
BATCH_SIZE      = 32
EPOCHS          = 30
LR              = 1e-4
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
DROPOUT_RATE    = 0.6
PATIENCE        = 8

# E-FSDA v3 hyperparams
EFSDA_REDUCTION  = 16
EFSDA_STRIP_K    = 7    # Strip conv kernel size

# Loss
FOCAL_GAMMA  = 2.0
CB_BETA      = 0.9999
ADAPTIVE_TAU = 0.3

N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

print(f"Strategy: {STRATEGY_LABEL}")
print(f"Novel: Multi-Scale Strip Spatial + Cross-Branch Gating + Gated Fusion")
print(f"Unfreeze: {UNFREEZE_BLOCKS} | LR: {LR} | BS: {BATCH_SIZE}")

In [ ]:
# ========== 3. E-FSDA v3: ENHANCED FSDA WITH CROSS-BRANCH GATING ========== #

class EnhancedFrequencyAttention(Layer):
    """Frequency Channel Attention with Temperature Scaling.
    All call() computation in float32; weights cast explicitly at use-site
    to survive mixed_float16 policy.
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.temperature = self.add_weight(
            name='freq_temp', shape=(1, C),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        x_t = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft = tf.signal.fft2d(x_complex)
        mag = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        attn = self.fc2(attn)  # float32 (fc2 has dtype='float32')
        # Cast weight to float32 at use-site — survives mixed_float16 policy
        temp = tf.nn.softplus(tf.cast(self.temperature, tf.float32)) + 1e-6
        attn = tf.nn.sigmoid(attn / temp)
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out = x_f32 * attn
        return tf.cast(out, x.dtype), attn

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class MultiScaleSpatialAttention(Layer):
    """Multi-Scale Spatial Attention with Strip Convolutions.
    Weights cast to float32 at use-site.
    """
    def __init__(self, strip_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.strip_kernel = strip_kernel

    def build(self, input_shape):
        self.conv3 = Conv2D(1, 3, padding='same', use_bias=False,
                            dtype='float32', name=f'{self.name}_conv3')
        self.conv5 = Conv2D(1, 5, padding='same', use_bias=False,
                            dtype='float32', name=f'{self.name}_conv5')
        self.strip_h = Conv2D(1, (1, self.strip_kernel), padding='same',
                              use_bias=False, dtype='float32', name=f'{self.name}_strip_h')
        self.strip_v = Conv2D(1, (self.strip_kernel, 1), padding='same',
                              use_bias=False, dtype='float32', name=f'{self.name}_strip_v')
        self.scale_weights = self.add_weight(
            name='scale_weights', shape=(4,),
            initializer=tf.keras.initializers.Constant(0.25),
            trainable=True)
        self.sp_temperature = self.add_weight(
            name='sp_temp', shape=(1, 1, 1, 1),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())
        in_shape = tuple(input_shape[:-1]) + (2,)
        self.conv3.build(in_shape)
        self.conv5.build(in_shape)
        self.strip_h.build(in_shape)
        self.strip_v.build(in_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32, axis=-1, keepdims=True)
        pooled = tf.concat([avg_pool, max_pool], axis=-1)

        s3 = self.conv3(pooled)
        s5 = self.conv5(pooled)
        sh = self.strip_h(pooled)
        sv = self.strip_v(pooled)

        # Cast weights to float32 at use-site
        w = tf.nn.softmax(tf.cast(self.scale_weights, tf.float32))
        fused = w[0]*s3 + w[1]*s5 + w[2]*sh + w[3]*sv

        temp = tf.nn.softplus(tf.cast(self.sp_temperature, tf.float32)) + 1e-6
        sp_attn = tf.nn.sigmoid(fused / temp)
        return sp_attn

    def get_config(self):
        cfg = super().get_config()
        cfg['strip_kernel'] = self.strip_kernel
        return cfg


class EFSDAv3Block(Layer):
    """Enhanced FSDA v3: Cross-Branch Gating + Multi-Scale Spatial + Gated Fusion."""
    def __init__(self, reduction=16, strip_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.strip_kernel = strip_kernel

    def build(self, input_shape):
        C = input_shape[-1]
        self.freq_attn = EnhancedFrequencyAttention(
            reduction=self.reduction, name=f'{self.name}_freq')
        self.spatial_attn = MultiScaleSpatialAttention(
            strip_kernel=self.strip_kernel, name=f'{self.name}_spatial')
        self.sp_to_freq_fc = Dense(C, activation='sigmoid', dtype='float32',
                                    name=f'{self.name}_sp2freq')
        self.freq_to_sp_conv = Conv2D(1, 1, padding='same', activation='sigmoid',
                                      dtype='float32', name=f'{self.name}_freq2sp')
        self.fusion_gate = Dense(1, activation='sigmoid', dtype='float32',
                                 name=f'{self.name}_fusion_gate')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        self.freq_attn.build(input_shape)
        self.spatial_attn.build(input_shape)
        self.sp_to_freq_fc.build((None, C))
        self.freq_to_sp_conv.build(tuple(input_shape[:-1]) + (C,))
        self.fusion_gate.build((None, C))
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        x_f32 = tf.cast(x, tf.float32)

        # Frequency Branch
        freq_out, freq_channel_attn = self.freq_attn(x, training=training)
        freq_out = tf.cast(freq_out, tf.float32)

        # Spatial Branch
        sp_attn = self.spatial_attn(x, training=training)  # float32
        spatial_out = x_f32 * sp_attn

        # Cross-Branch Gating
        sp_squeezed = tf.reduce_mean(sp_attn * x_f32, axis=[1, 2])
        sp_gate = self.sp_to_freq_fc(sp_squeezed)
        sp_gate = tf.reshape(sp_gate, [-1, 1, 1, tf.shape(x_f32)[-1]])
        freq_out = freq_out * sp_gate

        freq_spatial_hint = self.freq_to_sp_conv(freq_out)
        spatial_out = spatial_out * freq_spatial_hint

        # Gated Fusion
        gap_feat = tf.reduce_mean(x_f32, axis=[1, 2])
        gate = self.fusion_gate(gap_feat)
        gate = tf.reshape(gate, [-1, 1, 1, 1])

        fused = gate * freq_out + (1.0 - gate) * spatial_out
        fused = fused + x_f32  # residual
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        """Bypass tf.signal.fft2d symbolic tracing for Keras 3."""
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'strip_kernel': self.strip_kernel})
        return cfg


print("E-FSDA v3 defined.")
print("  Novel 1: Multi-Scale Strip Spatial Attention (3x3 + 5x5 + 1xk + kx1)")
print("  Novel 2: Cross-Branch Gating (freq<->spatial mutual information)")
print("  Novel 3: Learnable Gated Fusion + Residual Connection")
print("  Novel 4: Temperature Scaling on both branches")


In [ ]:
# ========== 4. ADAPTIVE CB FOCAL LOSS ========== #

class AdaptiveClassBalancedFocalLoss(tf.keras.losses.Loss):
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor')

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        combined_weights = self.static_weights * self.adaptive_factor
        combined_weights = combined_weights / tf.reduce_mean(combined_weights)
        sample_w = tf.reduce_sum(y_true * combined_weights, axis=-1)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


class AdaptiveWeightCallback(Callback):
    def __init__(self, loss_fn, val_ds, num_classes, class_names, tau=0.3, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn = loss_fn
        self.val_ds = val_ds
        self.num_classes = num_classes
        self.class_names = class_names
        self.tau = tau
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            per_class_recall[c] = (y_pred[mask] == c).mean() if mask.sum() > 0 else 1.0
        epsilon = 0.1
        adaptation_target = (1.0 - per_class_recall) + epsilon
        current_factor = self.loss_fn.adaptive_factor.numpy()
        new_factor = (1.0 - self.tau) * current_factor + self.tau * adaptation_target
        new_factor = new_factor / new_factor.mean()
        self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
        self.history.append({'epoch': epoch+1,
                             'per_class_recall': per_class_recall.copy(),
                             'adaptive_factor': new_factor.copy()})

print("Adaptive CB Focal Loss defined.")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    base.trainable = False
    for layer in base.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base.layers)} layers trainable")


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=dict(enumerate(cw)),
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helpers defined.")

In [ ]:

# ========== 6. MODEL BUILDER ========== #

CUSTOM_OBJECTS = {
    'EnhancedFrequencyAttention': EnhancedFrequencyAttention,
    'MultiScaleSpatialAttention': MultiScaleSpatialAttention,
    'EFSDAv3Block': EFSDAv3Block,
    'AdaptiveClassBalancedFocalLoss': AdaptiveClassBalancedFocalLoss,
}


def build_efsda_v3_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    """EfficientNetB4 + E-FSDA v3 (CrossBranch + MultiScale)."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output

    attended, sp_attn_map = EFSDAv3Block(
        reduction=EFSDA_REDUCTION,
        strip_kernel=EFSDA_STRIP_K,
        name='efsda_v3',
    )(feat_map)

    x = GlobalAveragePooling2D(name='gap')(attended)
    x = BatchNormalization(name='head_bn')(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out, name='EfficientNetB4_EFSDAv3')

    loss_fn = AdaptiveClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes, gamma=FOCAL_GAMMA, beta=CB_BETA)

    # Use plain float LR — ReduceLROnPlateau requires a settable scalar LR,
    # NOT a LearningRateSchedule object (the two are incompatible).
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
                  loss=loss_fn, metrics=['accuracy'])
    return model, loss_fn


print("Model builder defined: build_efsda_v3_model()")


In [ ]:

# ========== 7. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for cn in meta.class_names
    ])

    model, loss_fn = build_efsda_v3_model(
        INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)

    if run_idx == 0:
        model.summary(print_fn=lambda x: print(x) if 'efsda' in x.lower() or 'Total' in x or 'Trainable' in x else None)

    adaptive_cb = AdaptiveWeightCallback(
        loss_fn=loss_fn, val_ds=val_ds,
        num_classes=meta.num_classes, class_names=meta.class_names, tau=ADAPTIVE_TAU)

    callbacks = [
        adaptive_cb,
        # Monitor val_accuracy for early stopping + checkpoint — avoids saving overfit low-loss models
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                      restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    ]

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks)

    # Evaluate
    best_model = load_model(os.path.join(RESULT_DIR, 'best_model.keras'),
                            custom_objects=CUSTOM_OBJECTS)
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=meta.class_names, digits=4))

    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_title(f'CM — Run {run_idx+1}'); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300); plt.close()

    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train, 'n_val': meta.n_val, 'n_test': meta.n_test,
        'adaptive_history': adaptive_cb.history,
    })

    print(f"  Acc={test_acc:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    tf.keras.backend.clear_session()

print("\n" + "="*70 + f"\n ALL {N_RUNS} RUNS COMPLETED\n" + "="*70)


In [ ]:
# ========== 8. RESULTS AGGREGATION ========== #

accuracies = [r['accuracy'] for r in all_runs_results]
f1_scores  = [r['f1_score'] for r in all_runs_results]

print(f"\n{'='*60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'='*60}")
print(f"  Accuracy : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1-Score : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Per run  : {[f'{a:.4f}' for a in accuracies]}")

for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    print(f"  Run {r['run']}: Kappa={kappa:.4f}  MCC={mcc:.4f}")

summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print(f"\n✅ Archived → {zip_path}.zip")